In [1]:
# | eval: false

In [2]:
#| default_exp noising_time

In [3]:
#|export
import numpy as np
import torch


In [4]:
#| export
def set_seed(seed):
    np.random.seed(seed); torch.manual_seed(seed)

def cosine_beta_scheduler(T, s=0.008):
    t         = np.linspace(0, T, T + 1) / T
    alpha_bar = np.cos((t + s) / (1 + s) * np.pi / 2) ** 2
    alpha_bar = alpha_bar / alpha_bar[0]
    betas     = 1 - (alpha_bar[1:] / alpha_bar[:-1])
    return np.clip(betas, 1e-5, 0.999)

def timestep_embedding(t, dim=128):
    half = dim // 2
    emb  = torch.log(torch.tensor(10000., device=t.device)) / (half - 1)
    emb  = torch.exp(torch.arange(half, device=t.device) * -emb)
    emb  = t[:, None] * emb[None, :]
    return torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)

def noisify(T, x0, alphas_bar):
    t     = torch.randint(0, T, size=(x0.shape[0],), device=x0.device)
    eps   = torch.randn_like(x0)
    a_bar = alphas_bar[t].unsqueeze(1)
    xt    = torch.sqrt(a_bar) * x0 + torch.sqrt(1 - a_bar) * eps
    return xt, t, eps

def split_context(context):
    return context[:, :MARKET_DIM], context[:, MARKET_DIM:]

T          = 100
betas      = cosine_beta_scheduler(T)
alphas     = 1 - betas
alphas_bar = torch.cumprod(torch.from_numpy(alphas).float(), dim=0)
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
alphas_bar = alphas_bar.to(device)
print(f"Device: {device} | T={T}")

Device: cpu | T=100
